# FLUX.2 Klein Image Generation with OpenVINO™

[FLUX.2 [klein]](https://huggingface.co/black-forest-labs/FLUX.2-klein-4B) is a 4-billion-parameter rectified flow transformer from Black Forest Labs.
It unifies text-to-image generation and image editing in a single compact architecture,
delivering state-of-the-art quality with end-to-end inference in as low as 4 steps.

**Key features:**
- Sub-second generation with distilled 4-step mode
- Text-to-image and multi-reference image editing in one model
- Runs on consumer GPUs (~13 GB VRAM)
- Open weights under Apache 2.0 license

In this notebook we demonstrate how to convert and optimize FLUX.2 [klein] 4B using OpenVINO with INT4 weight compression.

> **Note**: This notebook requires at least 32 GB RAM for model conversion and ~16 GB for INT4 inference.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert model with OpenVINO](#Convert-model-with-OpenVINO)
  - [Select weight format](#Select-weight-format)
  - [Export model using Optimum Intel](#Export-model-using-Optimum-Intel)
- [Run OpenVINO model inference](#Run-OpenVINO-model-inference)
  - [Text-to-Image](#Text-to-Image)
- [Interactive demo](#Interactive-demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/flux.2-klein/flux.2-klein.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform
import requests
from pathlib import Path

for fname in ("notebook_utils.py", "cmd_helper.py", "pip_helper.py"):
    if not Path(fname).exists():
        r = requests.get(
            url=f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{fname}",
        )
        open(fname, "w").write(r.text)

from pip_helper import pip_install

%pip uninstall -q -y diffusers optimum-intel

pip_install(
    "-q",
    "gradio>=4.19,<6",
    "torch>=2.4",
    "nncf>=2.15.0",
    "accelerate",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)
pip_install(\"-q\", \"git+https://github.com/huggingface/diffusers.git@153fcbc5a81be884304dcbe64986b7ea917a3802\")
pip_install("-q", "git+https://github.com/openvino-dev-samples/optimum-intel.git@flux.2-klein")
pip_install("-qU", "openvino>=2026.0")

if platform.system() == "Darwin":
    pip_install("numpy<2.0.0")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("flux.2-klein.ipynb")

## Convert model with OpenVINO
[back to top ⬆️](#Table-of-contents:)

FLUX.2 [klein] 4B uses a rectified flow transformer architecture. The pipeline consists of:

* **Text Encoder** — Qwen3 language model that creates conditioning embeddings from text prompts (stacked hidden states from layers 9, 18, 27).
* **Transformer** — Flux2Transformer2DModel (4B parameters) for step-by-step denoising of the latent image representation.
* **VAE** — AutoencoderKLFlux2 for encoding/decoding between pixel and latent space.

We use [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) to export all components to OpenVINO IR format.

### Select weight format
[back to top ⬆️](#Table-of-contents:)

INT4 weight compression significantly reduces model size and improves inference speed with minimal quality loss.

In [ ]:
import ipywidgets as widgets

model_id = "black-forest-labs/FLUX.2-klein-4B"

to_compress = widgets.Checkbox(
    value=True,
    description="INT4 weight compression",
    disabled=False,
)

to_compress

### Export model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

[Optimum Intel](https://huggingface.co/docs/optimum/intel/index) provides `optimum-cli` for model export.

In [ ]:
from pathlib import Path

model_base_dir = Path("FLUX.2-klein-4B")
additional_args = {"task": "text-to-image"}

if to_compress.value:
    model_dir = model_base_dir / "INT4"
    additional_args.update({"weight-format": "int4", "ratio": "0.8"})
else:
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

print(f"Model will be exported to: {model_dir}")
print(f"Weight format: {'INT4' if to_compress.value else 'FP16'}")

In [ ]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)
else:
    print(f"Model already exists at {model_dir}, skipping export.")

## Run OpenVINO model inference
[back to top ⬆️](#Table-of-contents:)

Select the device for running inference using OpenVINO.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

In [ ]:
from optimum.intel import OVFlux2KleinPipeline
import torch

ov_config = {}
if "GPU" in device.value:
    # Scale factor prevents FP16 overflow on Intel Arc GPUs
    ov_config["ACTIVATIONS_SCALE_FACTOR"] = "8.0"

ov_pipe = OVFlux2KleinPipeline.from_pretrained(
    model_dir,
    device=device.value,
    ov_config=ov_config,
)
print(f"Pipeline loaded from {model_dir}")

### Text-to-Image
[back to top ⬆️](#Table-of-contents:)

Generate an image from a text prompt. FLUX.2 Klein (distilled) works best with `guidance_scale=1.0` and `num_inference_steps=4`.

In [ ]:
prompt = "A cat holding a sign that says hello world"

generator = torch.Generator("cpu").manual_seed(0)

result = ov_pipe(
    prompt=prompt,
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=4,
    generator=generator,
)

result.images[0]

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_pipe)

try:
    demo.launch(debug=False)
except Exception:
    demo.launch(share=True, debug=False)

# If you are launching remotely, specify server_name and server_port:
# demo.launch(server_name='your server name', server_port='server port number')